In [13]:
SEGMENT_SIZE = 2
MAX_N_SEGMENTS = 2


In [14]:
# If needed, install deps (uncomment to auto-install)
# !pip -q install transformers datasets accelerate peft git+https://github.com/ai-forever/lm-experiments-tools

import os
import logging
from itertools import chain
from pathlib import Path

import torch
import numpy as np
import datasets

from torch.nn.utils.rnn import pad_sequence
import accelerate
from transformers import (
    AutoConfig,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)

# Utility to resolve "module:ClassName" dynamically (compatible with lm_experiments_tools.utils.get_cls_by_name)
def get_cls_by_name(path: str):
    if ":" not in path:
        raise ValueError(f"Expected 'module:ClassName' format, got {path}")
    module_name, cls_name = path.split(":")
    module = __import__(module_name, fromlist=[cls_name])
    return getattr(module, cls_name)

# ---------------------------
# Configuration (edit here)
# ---------------------------
class Args:
    # Data
    task_name = "HuggingFaceFW/fineweb-edu"
    tokenized_dataset = None
    validate_only = False
    working_dir = "."
    show_valid_examples = 5
    sample_size = SEGMENT_SIZE * MAX_N_SEGMENTS
    val_sample_size = SEGMENT_SIZE * MAX_N_SEGMENTS
    data_n_workers = 2
    input_prefix = ""
    sliding_window = False

    # Model
    from_pretrained = "HuggingFaceTB/SmolLM2-135M"
    model_cfg = None
    model_cls = "transformers:AutoModelForCausalLM"
    memory_cell_cls = "modeling_rmt.language_modeling:MemoryCell"
    recurrent_wrapper_cls = "modeling_rmt.language_modeling:RecurrentWrapper"
    model_cpt = None
    checkpoint = None
    model_type = "decoder"

    # RMT
    segment_size = SEGMENT_SIZE
    num_mem_tokens = 16
    max_n_segments = MAX_N_SEGMENTS
    vary_n_segments = False
    loss_from_last_seg_only = False
    no_loss_from_first_segment = True
    min_sample_len = 16000
    sum_loss = False
    bptt_depth = -1
    segment_ordering = "regular"
    retain_graph = False
    use_truncated_backward = False
    k1 = -1
    k2 = -1
    freeze_model_weights = False
    backbone_cpt = None

    # Tokenizer
    tokenizer = None

    # Optimizer-related
    optimizer = "AdamW"
    scale_parameter = False
    relative_step = False
    warmup_init = False

    # LoRA (disabled by default)
    use_lora = False
    lora_attn_dim = 8
    lora_attn_alpha = 32
    lora_dropout = 0.1

    # TrainingArguments (set as in script)
    output_dir = "./outputs"
    per_device_train_batch_size = 1
    num_train_epochs = None  # Not set in script; using max_steps instead
    learning_rate = 3e-4
    logging_steps = 25
    save_steps = None  # Not set in script; using save_total_limit=1
    eval_steps = 100
    gradient_accumulation_steps = 1024  # TBS/(BS*NP) = 1024/(1*1)
    fp16 = torch.cuda.is_available()

args = Args()

# ---------------------------
# Logging and CUDA visibility
# ---------------------------
logger_fmt = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logging.basicConfig(format=logger_fmt, level=logging.INFO)
logger = logging.getLogger('')

if os.environ.get('CUDA_VISIBLE_DEVICES', None) is None:
    os.environ['CUDA_VISIBLE_DEVICES'] = ','.join([str(i) for i in range(torch.cuda.device_count())])

logger.info(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")
logger.info(f"CUDA DEVICE COUNT: {torch.cuda.device_count()}")

2025-08-27 19:23:30,533 - root - INFO - CUDA_VISIBLE_DEVICES: 0
2025-08-27 19:23:30,535 - root - INFO - CUDA DEVICE COUNT: 1


In [15]:

# ---------------------------
# Working dir and accelerator
# ---------------------------
args.working_dir = str(Path(args.working_dir).expanduser().absolute())
os.chdir(args.working_dir)

accelerator = accelerate.Accelerator(gradient_accumulation_steps=args.gradient_accumulation_steps)
from accelerate.logging import get_logger as _get_accel_logger
logger = _get_accel_logger('')
logger.info(f'num processes: {accelerator.num_processes}')
logger.info(f'mixed precision: {accelerator.mixed_precision}')

# ---------------------------
# Tokenizer
# ---------------------------
if args.tokenizer:
    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer)
else:
    tokenizer = AutoTokenizer.from_pretrained(args.from_pretrained)

# Ensure pad token exists
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------------------------
# Prepare datasets
# ---------------------------
logger.info(f'preparing dataset for {args.task_name}')

segment_size = args.segment_size
history_size = args.sample_size - segment_size

if args.val_sample_size is not None:
    val_history_size = args.val_sample_size - segment_size
else:
    val_history_size = history_size

def group_texts(examples, segment_size, history_size=None):
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    if history_size is None:
        result = {
            k: [t[i: i + segment_size] for i in range(0, total_length, segment_size)]
            for k, t in concatenated_examples.items()
        }
    else:
        result = {
            k: [t[max({0, i - history_size}): i + segment_size]
                for i in range(history_size, total_length, segment_size)]
            for k, t in concatenated_examples.items()
        }
    return result

id_pad_value = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def collate_fn(batch):
    input_ids = labels = [torch.tensor(b['input_ids']) for b in batch]
    attention_mask = [torch.ones_like(b, dtype=int) for b in input_ids]
    labels_mask = [torch.ones_like(b, dtype=int) for b in input_ids]

    if getattr(args, 'loss_from_last_seg_only', False):
        for m in labels_mask:
            m[:-args.segment_size] = False

    if getattr(args, 'no_loss_from_first_segment', False):
        for m in labels_mask:
            m[:args.segment_size] = False

    input_ids = pad_sequence(input_ids, padding_value=id_pad_value, batch_first=True)
    labels = pad_sequence(labels, padding_value=-100, batch_first=True)
    attention_mask = pad_sequence(attention_mask, padding_value=0, batch_first=True)
    labels_mask = pad_sequence(labels_mask, padding_value=0, batch_first=True)

    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': attention_mask,
        'labels_mask': labels_mask.bool()
    }

2025-08-27 19:23:31,068 - root - INFO - num processes: 1
2025-08-27 19:23:31,068 - root - INFO - mixed precision: no
2025-08-27 19:23:31,864 - root - INFO - preparing dataset for HuggingFaceFW/fineweb-edu


In [16]:

with accelerator.main_process_first():
    if args.tokenized_dataset is not None:
        dataset = datasets.load_from_disk(args.tokenized_dataset)
    elif args.task_name is not None:
        if 'fineweb' in args.task_name:
            train_dataset = datasets.load_dataset("HuggingFaceFW/fineweb-edu", name="CC-MAIN-2024-10", split="train", streaming=True)
            other_dataset = datasets.load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1')
            valid_dataset = other_dataset["validation"]
            test_dataset = other_dataset["test"]
        else:
            if 'wikitext' in args.task_name:
                dataset = datasets.load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1')
            else:
                dataset = datasets.load_dataset(args.task_name)
            train_dataset = dataset['train']
            valid_dataset = dataset["validation"]
            test_dataset = dataset["test"]

        # Tokenize text column
        train_dataset = train_dataset.map(lambda x: tokenizer(x['text'], add_special_tokens=False),
                                          batched=True, batch_size=50_000)
        valid_dataset = valid_dataset.map(lambda x: tokenizer(x['text'], add_special_tokens=False),
                                          batched=True, batch_size=50_000)
        test_dataset = test_dataset.map(lambda x: tokenizer(x['text'], add_special_tokens=False),
                                        batched=True, batch_size=50_000)
    else:
        raise NotImplementedError("Provide either tokenized_dataset or task_name")

with accelerator.main_process_first():
    train_dataset = train_dataset.select_columns(['input_ids']).map(
        lambda x: group_texts(x, segment_size, history_size), batched=True, batch_size=50_000)
    valid_dataset = valid_dataset.select_columns(['input_ids']).map(
        lambda x: group_texts(x, segment_size, val_history_size), batched=True, batch_size=50_000)
    test_dataset = test_dataset.select_columns(['input_ids']).map(
        lambda x: group_texts(x, segment_size, val_history_size), batched=True, batch_size=50_000)

# Subsample validation for speed
num_valid_examples = 100
valid_inds = np.linspace(1, len(valid_dataset)-1, num_valid_examples).astype(int).tolist()
valid_dataset = valid_dataset.select(valid_inds)

Map: 100%|██████████| 4358/4358 [00:00<00:00, 6148.78 examples/s]


In [20]:
gen = iter(train_dataset)
batch = next(gen)
len(batch['input_ids'])

Token indices sequence length is longer than the specified maximum sequence length for this model (11007 > 8192). Running this sequence through the model will result in indexing errors


4

In [21]:

tokenizer.batch_decode(batch['input_ids'])

['-', ' It', ' means', ' objects']

In [19]:
len(train_dataset[0]['input_ids'])

TypeError: object of type 'IterableColumn' has no len()

In [18]:
tokenizer.batch_decode(valid_dataset['input_ids'])

['arus gammarus',
 ' Training School ( F',
 ' year before , Wil',
 ' Chōji .',
 ' area . Between ',
 ' Mississippi State Hospital in',
 'uderdale County .',
 ' 195',
 ' daily newspaper since ',
 ' for his plans ,',
 ' A request had been',
 ' of the most monumental',
 'iving his first start',
 ' states canceled classes for',
 ' party " Missh',
 '75 , US',
 ' Richmond and his father',
 ' hospitalized in West Palm',
 ' annually . Much of',
 ' was found to be',
 ' " . \n The',
 '@ American ( third',
 'ify the character ,',
 ' sixth . Radcliffe',
 ' the river near historic',
 ' Parkway took up',
 ' only way forward .',
 ' captured en route and',
 ' move toward Guadal',
 '00 , but',
 " from Tamura '",
 ' , with CR ',
 " Warsaw Pact 's",
 ' this ill @-',
 ' ’ s abandonment by',
 ' to the south .',
 ' Spanish friar And',
 ' Jasaw Chan',
 '869 by',
 ' a band of convection',
 ' none of the conventional',
 ' of Scientology as',
 ' press saying they are',
 ' with United States National',
 ' 5 @-',

In [ ]:

# ---------------------------
# Define / init model
# ---------------------------
model_cls = get_cls_by_name(args.model_cls)
logging.getLogger('').info(f'Using model class: {model_cls}')

if not args.from_pretrained:
    if args.model_cfg is None:
        raise ValueError("Either set Args.from_pretrained or Args.model_cfg")
    model_cfg = AutoConfig.from_pretrained(args.model_cfg)
    model = model_cls.from_config(config=model_cfg)
else:
    logging.getLogger('').info(f'Loading pretrained model: {args.from_pretrained}')
    model = model_cls.from_pretrained(args.from_pretrained)

# Optional LoRA
if getattr(args, "use_lora", False):
    from peft import LoraConfig, TaskType, get_peft_model
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=args.lora_attn_dim,
        lora_alpha=args.lora_attn_alpha,
        lora_dropout=args.lora_dropout
    )
    model = get_peft_model(model, peft_config)
    logging.getLogger('').info('Added LoRA, trainable parameters with LoRA only')
    try:
        model.print_trainable_parameters()
    except Exception:
        pass

# Optional backbone checkpoint
if args.backbone_cpt:
    cpt = torch.load(args.backbone_cpt, map_location='cpu')
    model.load_state_dict(cpt['model_state_dict'], strict=False)
    logging.getLogger('').info(f'Loaded baseline state dict from: {args.backbone_cpt}')

# Optional RMT wrapping
if args.num_mem_tokens is not None:
    if args.memory_cell_cls is None or args.recurrent_wrapper_cls is None:
        raise ValueError("Provide both memory_cell_cls and recurrent_wrapper_cls when num_mem_tokens is set.")
    memory_cell_cls = get_cls_by_name(args.memory_cell_cls)
    recurrent_wrapper_cls = get_cls_by_name(args.recurrent_wrapper_cls)
    logging.getLogger('').info(f'Wrapping in: {memory_cell_cls} and {recurrent_wrapper_cls}')

    cell = memory_cell_cls(model, num_mem_tokens=args.num_mem_tokens)
    model = recurrent_wrapper_cls(
        cell,
        segment_size=segment_size,
        max_n_segments=int(args.max_n_segments),
        vary_n_segments=args.vary_n_segments,
        k2=args.k2
    )

    if args.model_cpt and args.model_cpt != 'None':
        cpt = torch.load(args.model_cpt, map_location='cpu')
        model.load_state_dict(cpt, strict=False)
        logging.getLogger('').info(f'Loaded RMT state dict from: {args.model_cpt}')

# ---------------------------
# TrainingArguments
# ---------------------------
# Start with a TrainingArguments instance to probe valid keys
_probing = TrainingArguments(output_dir=args.output_dir, per_device_train_batch_size=args.per_device_train_batch_size)
valid_keys = set(vars(_probing).keys())

# Mimic original behavior: gather keys from args if present in TrainingArguments
training_args_dict = {k: getattr(args, k) for k in dir(args) if (not k.startswith('_') and k in valid_keys)}
training_args_dict.update({
    'output_dir': args.output_dir,
    'per_device_train_batch_size': args.per_device_train_batch_size,
    'num_train_epochs': args.num_train_epochs,
    'learning_rate': args.learning_rate,
    'logging_steps': args.logging_steps,
    'save_steps': args.save_steps,
    'eval_steps': args.eval_steps,
    'evaluation_strategy': 'steps',
    'remove_unused_columns': False,
    'save_safetensors': False,
    'label_names': ['labels'],
    'gradient_accumulation_steps': args.gradient_accumulation_steps,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'log_level': 'debug',
    'ignore_data_skip': True,
    'warmup_steps': 1000,
})

# Precision flags (prefer fp16 default here; set bf16 only if supported)
training_args_dict['fp16'] = getattr(args, 'fp16', False)
if torch.cuda.is_available():
    # enable bf16 on Ampere+ if you prefer (optional)
    compute_capability = torch.cuda.get_device_capability(0)
    is_ampere_or_newer = compute_capability[0] >= 8
    training_args_dict['bf16'] = bool(is_ampere_or_newer)

# Derive eval batch size if not specified
if 'per_device_eval_batch_size' not in training_args_dict:
    training_args_dict['per_device_eval_batch_size'] = max(1, training_args_dict['per_device_train_batch_size'] // 4)

# Reasonable default for eval accumulation
training_args_dict['eval_accumulation_steps'] = 32

training_args = TrainingArguments(**training_args_dict)

# ---------------------------
# Trainer and run
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collate_fn,
)

print("Trainer Gradient Checkpointing Enabled:", trainer.args.gradient_checkpointing)

if not args.validate_only:
    trainer.train(resume_from_checkpoint=args.checkpoint)
else:
    metrics = trainer.evaluate()
    print(metrics)